# Ders 7: Finansal Zaman Serileri
Bu notebook'ta şunları öğreneceğiz:
- Finansal getirilerin istatistiksel özellikleri
- ARCH ve GARCH modelleri (koşullu değişken varyans)
- EGARCH — asimetrik volatilite (kaldıraç etkisi)
- Geometrik Brownian Motion ve Black-Scholes


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# arch kütüphanesi
try:
    from arch import arch_model
    print("arch kütüphanesi mevcut.")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'arch', '-q'])
    from arch import arch_model
    print("arch kütüphanesi kuruldu.")

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
print("Hazır.")


## 7.1 Finansal Getirilerin Özellikleri

**Log getiri:** $r_t = \log(P_t/P_{t-1}) = \log P_t - \log P_{t-1}$

**Stilize Olgular (Stylized Facts):**
1. **Ağır kuyruklar** — Normal dağılımdan fazla ekstrem değer
2. **Volatilite kümelenmesi** — Yüksek volatilite dönemleri peş peşe gelir
3. **Kaldıraç etkisi** — Negatif şoklar volatiliteyi daha çok artırır
4. **Otokorelasyon** — Getirilerde yok, kare getirilerde var


In [ ]:
# ── Gerçekçi Hisse Senedi Fiyat Simülasyonu (GBM + GARCH volatilite) ──

np.random.seed(42)
n = 1000

# GARCH(1,1) volatilite süreci
omega, alpha, beta = 0.00001, 0.10, 0.85
sigma2 = np.zeros(n)
sigma2[0] = omega / (1 - alpha - beta)
eps = np.zeros(n)
r = np.zeros(n)
mu = 0.0003  # günlük ortalama getiri

for t in range(1, n):
    eps[t] = np.random.normal(0, np.sqrt(sigma2[t-1]))
    r[t] = mu + eps[t]
    sigma2[t] = omega + alpha * eps[t]**2 + beta * sigma2[t-1]

# Fiyat serisi
P = 100 * np.exp(np.cumsum(r))
sigma_t = np.sqrt(sigma2)

fig, axes = plt.subplots(3, 2, figsize=(14, 10))

# Fiyat
axes[0,0].plot(P, color='steelblue', lw=0.8)
axes[0,0].set_title('Simüle Fiyat Serisi', fontweight='bold')
axes[0,0].set_ylabel('Fiyat')

# Getiriler
axes[0,1].plot(r, color='darkorange', lw=0.5, alpha=0.7)
axes[0,1].set_title('Log Getiriler $r_t$', fontweight='bold')

# Volatilite
axes[1,0].plot(sigma_t * np.sqrt(252), color='red', lw=0.8)
axes[1,0].set_title('Gerçek Volatilite (Yıllık %)', fontweight='bold')

# Kare getiriler (volatilite kümelenmesi)
axes[1,1].plot(r**2, color='purple', lw=0.5, alpha=0.8)
axes[1,1].set_title('$r_t^2$ — Volatilite Kümelenmesi', fontweight='bold')

# ACF
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(r, lags=20, ax=axes[2,0], title='ACF($r_t$) — Otokorelasyon Yok')
plot_acf(r**2, lags=20, ax=axes[2,1], title='ACF($r_t^2$) — Otokorelasyon VAR!')

plt.tight_layout()
plt.show()

# Normallik testi
stat, p = stats.shapiro(r[:500])
print(f"Shapiro-Wilk p-degeri: {p:.6f} -> {'Normal degil' if p<0.05 else 'Normal'}")
kurt = stats.kurtosis(r)
print(f"Basiklik (excess kurtosis): {kurt:.3f} (Normal=0, kalın kuyruk>0)")


## 7.2 ARCH Modeli

**ARCH(q) — AutoRegressive Conditional Heteroskedasticity** (Engle, 1982):

$$r_t = \sigma_t \epsilon_t, \quad \epsilon_t \sim WN(0,1)$$
$$\sigma_t^2 = \omega + \alpha_1 r_{t-1}^2 + \cdots + \alpha_q r_{t-q}^2$$

**ARCH Etkisi Testi:** $r_t^2$ serisinde Ljung-Box testi.


In [ ]:
# ── ARCH Etkisi Testi ──

lb_r = acorr_ljungbox(r, lags=[5, 10, 20], return_df=True)
lb_r2 = acorr_ljungbox(r**2, lags=[5, 10, 20], return_df=True)

print("Ljung-Box Testi — r_t (getiriler):")
print(lb_r[['lb_stat', 'lb_pvalue']].round(4))

print("\nLjung-Box Testi — r_t^2 (kare getiriler):")
print(lb_r2[['lb_stat', 'lb_pvalue']].round(4))
print("\n=> r_t^2 icin p<0.05: ARCH etkisi var!")


## 7.3 GARCH(1,1) Modeli

**GARCH(p,q) — Generalized ARCH** (Bollerslev, 1986):

$$\sigma_t^2 = \omega + \sum_{i=1}^{q}\alpha_i r_{t-i}^2 + \sum_{j=1}^{p}\beta_j \sigma_{t-j}^2$$

**En yaygın: GARCH(1,1):**
$$\sigma_t^2 = \omega + \alpha r_{t-1}^2 + \beta \sigma_{t-1}^2$$

**Koşullar:**
- $\omega > 0$, $\alpha \geq 0$, $\beta \geq 0$
- $\alpha + \beta < 1$ (kovaryans durağanlığı)
- $\alpha + \beta \approx 1$ → **IGARCH** (volatilite şokları kalıcı)


In [ ]:
# ── GARCH(1,1) Model Tahmini ──

# arch paketi 100x ölçeklenmiş getiri bekler
r_scaled = r * 100

garch_model = arch_model(r_scaled, vol='Garch', p=1, q=1, mean='Constant', dist='Normal')
garch_fit = garch_model.fit(disp='off')

print(garch_fit.summary())
print(f"\nalpha + beta = {garch_fit.params['alpha[1]'] + garch_fit.params['beta[1]']:.4f}")
print("(1'e yakin: yuksek volatilite kalimliligi)")


In [ ]:
# ── GARCH(1,1): Tahmin Edilen vs Gerçek Volatilite ──

cond_vol = garch_fit.conditional_volatility / 100  # Ölçeği geri al

fig, axes = plt.subplots(2, 1, figsize=(13, 7))

axes[0].plot(r, color='gray', lw=0.5, alpha=0.6, label='Getiri')
axes[0].plot(2*cond_vol, color='red', lw=1.2, alpha=0.8, label='+2σ')
axes[0].plot(-2*cond_vol, color='red', lw=1.2, alpha=0.8, label='-2σ')
axes[0].set_title('Log Getiriler ve GARCH(1,1) Volatilite Bandı', fontweight='bold')
axes[0].legend()

axes[1].plot(sigma_t, color='steelblue', lw=1.2, label='Gerçek σ_t', alpha=0.8)
axes[1].plot(cond_vol, color='red', lw=1, ls='--', label='GARCH Tahmini σ_t', alpha=0.8)
axes[1].set_title('Gerçek vs Tahmin Edilen Volatilite', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

n_min = min(len(sigma_t), len(cond_vol))
corr = np.corrcoef(sigma_t[:n_min], cond_vol[:n_min])[0,1]
print(f"Gerçek ve tahmin edilen volatilite korelasyonu: {corr:.3f}")


## 7.4 EGARCH — Asimetrik Volatilite

**EGARCH(1,1)** (Nelson, 1991):

$$\log\sigma_t^2 = \omega + \alpha\left(\frac{|\epsilon_{t-1}|}{\sigma_{t-1}} - \sqrt{2/\pi}\right) + \gamma\frac{\epsilon_{t-1}}{\sigma_{t-1}} + \beta\log\sigma_{t-1}^2$$

**Kaldıraç etkisi:** $\gamma < 0$ ise negatif şoklar volatiliteyi daha çok artırır (hisse senetlerinde tipik).


In [ ]:
# ── EGARCH Kaldıraç Etkisi ──

egarch_model = arch_model(r_scaled, vol='EGARCH', p=1, q=1, mean='Constant')
egarch_fit = egarch_model.fit(disp='off')

print("EGARCH Parametreleri:")
for p, v, pv in zip(egarch_fit.params.index,
                     egarch_fit.params.values,
                     egarch_fit.pvalues.values):
    sig = "***" if pv < 0.01 else ("**" if pv < 0.05 else ("*" if pv < 0.1 else ""))
    print(f"  {p:15s}: {v:8.5f}  (p={pv:.4f}) {sig}")

gamma = egarch_fit.params.get('gamma[1]', egarch_fit.params.get('gamma', None))
if gamma is not None:
    print(f"\nKaldirac etkisi (gamma): {gamma:.4f}")
    print(f"{'Negatif => Kaldıraç etkisi VAR (tipik hisse senedi davranışı)' if gamma<0 else 'Pozitif => Ters kaldıraç'}")


## 7.5 Geometrik Brownian Motion (GBM) ve Black-Scholes

**GBM Stokastik Diferansiyel Denklemi:**
$$dS_t = \mu S_t\, dt + \sigma S_t\, dW_t$$

**Çözüm:**
$$S_t = S_0 \exp\left[(\mu - \tfrac{\sigma^2}{2})t + \sigma W_t\right]$$

**Black-Scholes Alım Hakkı Fiyatı:**
$$C = S_0 N(d_1) - K e^{-rT} N(d_2)$$
$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$


In [ ]:
# ── GBM Simülasyonu + Black-Scholes ──

from scipy.stats import norm

def gbm_sim(S0, mu, sigma, T, n_steps, n_paths=5, seed=42):
    np.random.seed(seed)
    dt = T / n_steps
    t = np.linspace(0, T, n_steps+1)
    paths = np.zeros((n_paths, n_steps+1))
    paths[:, 0] = S0
    for i in range(1, n_steps+1):
        Z = np.random.normal(0, 1, n_paths)
        paths[:, i] = paths[:, i-1] * np.exp((mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z)
    return t, paths

def black_scholes_call(S0, K, r, sigma, T):
    d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S0*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2), d1, d2

# Parametreler
S0, mu_gbm, sigma_gbm, T_year = 100, 0.10, 0.25, 1.0
t, paths = gbm_sim(S0, mu_gbm, sigma_gbm, T_year, n_steps=252, n_paths=20)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# GBM yolları
for i in range(20):
    axes[0].plot(t, paths[i], lw=0.7, alpha=0.6)
axes[0].axhline(S0, color='black', ls='--', label=f'S₀={S0}')
axes[0].set_title(f'GBM Simülasyonu (μ={mu_gbm}, σ={sigma_gbm})', fontweight='bold')
axes[0].set_xlabel('Zaman (yıl)')
axes[0].set_ylabel('Fiyat')
axes[0].legend()

# Black-Scholes: farklı K için opsiyon değeri
K_vals = np.linspace(70, 130, 50)
r_rf = 0.05
C_vals = [black_scholes_call(S0, K, r_rf, sigma_gbm, T_year)[0] for K in K_vals]
axes[1].plot(K_vals, C_vals, color='steelblue', lw=2)
axes[1].axvline(S0, color='red', ls='--', label=f'S₀={S0} (ATM)')
axes[1].set_title('Black-Scholes Alım Hakkı Değeri', fontweight='bold')
axes[1].set_xlabel('Kullanım Fiyatı K')
axes[1].set_ylabel('Opsiyon Değeri C')
axes[1].legend()

plt.tight_layout()
plt.show()

# Örnek hesaplama
C, d1, d2 = black_scholes_call(S0, 100, r_rf, sigma_gbm, T_year)
print(f"ATM Alim Hakki (S0=K=100): C = {C:.4f} TL")
print(f"d1={d1:.4f}, d2={d2:.4f}")
print(f"Delta = N(d1) = {norm.cdf(d1):.4f}")


## Özet

| Model | Kullanım |
|-------|----------|
| **ARCH(q)** | Basit koşullu değişken varyans |
| **GARCH(1,1)** | Standart volatilite modeli |
| **EGARCH** | Asimetrik/kaldıraç etkisi |
| **GBM** | Sürekli zamanlı fiyat modeli |
| **Black-Scholes** | Opsiyon fiyatlama |

**Temel kavramlar:**
- Volatilite kümelenmesi → ARCH/GARCH
- $\alpha + \beta \to 1$ → Volatilite şokları uzun süre etkili
- Kaldıraç etkisi ($\gamma < 0$) → EGARCH
